In [1]:
import sys

sys.path.append('../../../')

In [6]:
from __future__ import annotations

import copy
import warnings
from typing import Union, Optional, Sequence

import torch
import torch.nn as nn
import torch.utils.checkpoint as cp
from torch.nn.modules.utils import _ntuple, _triple

%load_ext autoreload
%autoreload 2

from computer_vision.slowfast.mmcv.cnn.bricks.conv_module import ConvModule
from computer_vision.slowfast.mmcv.cnn.brick import build_activation_layer
from computer_vision.slowfast.mmaction.models.backbones.resnet3d import BasicBlock3d, Bottleneck3d

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
class ResNet3d(nn.Module):
    """ResNet 3d backbone
    Args:
        depth (int): Depth of resnet, from (18,34,50,101,152}. Default to 50
        pretrained (str, optional): Name of pretrained model. Default to None
        stage_blocks (tuple, optional): Number of residual blocks in each stage. Default to None
        pretrained2d (bool): Whether to load pretrained 2D model. Default to True
        in_channels (int): Channel number of input features. Default to 3
        num_stages (int): Number of stages. Default to 4
        base_channels (int): Channel number of stem output features. Default to 64
        out_indices (sequence[int]): Indices of output features. Default to ``(3,)``
        spatial_strides (sequence[int]): Spatial strides of residual blocks of each stage. Default to ``(1,2,2,2)``
        temporal_strides (sequence[int]): Temporal strides of residual blocks of each stage. Default to ``(1,1,1,1)``
        dilations (sequence[int]): Dilation of each stage. Default to ``(1,1,1,1)``
        conv1_kernel (sequence[int]): Kernel size of the first conv layer. Default to ``(3,7,7)``
        conv1_stride_s (int): Spatial stride of the first conv layer. Default to 2
        conv1_stride_t (int): Temporal stride of the first conv layer. Defaul to 1
        pool1_stride_s (int): Spatial stride of the first pooling layer. Default to 2
        pool1_stride_t (int): Temporal stride of the first pooling layer. Default to 1
        with_pool2 (bool): Whether to use pool2. Default to True
        style (str): 'pytorch' or 'caffe'. If set to 'pytorch', the stride-tow layer is the 3x3 conv layer; otherwise, 
            the stride-two layer is the first 1x1 conv layer. Default to 'pytorch'
        frozen_stages (int): Stages to be frozen (all param fixed). -1 means not freezing any parameters. Default to -1
        inflate (sequence[int]): Inflate dimensions of each block. Default to ``(1,1,1,1)``
        inflate_style (str): '3x1x1' or '3x3x3' which determines the kernel sizes and padding strides for conv1 and conv2 in each
            block. Default to '3x1x1'
        conv_cfg (dict): Config for conv layer. Required keys is ``type``. Default to dict(type='Conv3d')
        norm_cfg (dict): Config for norm layer. Required keys are ``type`` and ``requires_grad``. Defaul to dict(type='BN3d', requires_grad=True)
        act_cfg (dict): Config dict for activation layer. Default to dict(type='ReLU', inplace=True)
        norm_eval (bool): Whether to set BN layers to eval mode, namely, freeze running stats (``mean`` and ``var``). Default to False
        with_cp (bool): Whether to use checkpoint or not. Using checkpoint will save some memory while slowing down the training speed.
            Default to False
        non_local (sequence[int]): Determine whether to apply non-local module in the corresponding block of each stage. Default to ``(0,0,0,0)``
        zero_init_residual (bool): Whether to use zero initialization for residual block. Defaul to True
        init_cfg (dict|list[dict],optional): Initialization config dict. Default to None
    """
    arch_settings={18:(BasicBlock3d, (2,2,2,2)),
                  34:(BasicBlock3d, (3,4,6,3)),
                  50:(Bottleneck3d, (3,4,6,3)),
                  101:(Bottleneck3d, (3,4,23,3)),
                  152:(Bottleneck3d, (3,8,36,3))}
    def __init__(self, depth:int=50, pretrained:Optional[str]=None, stage_blocks:Optional[tuple]=None,
                 pretrained2d:bool=True, in_channels:int=3, num_stages:int=4, base_channels:int=64,
                 out_indices:Sequence[int]=(3,), spatial_strides:Sequence[int]=(1,2,2,2), 
                 temporal_strides:Sequence[int]=(1,1,1,1), dilations:Sequence[int]=(1,1,1,1), 
                 conv1_kernel:Sequence[int]=(3,7,7), conv1_stride_s:int=2, conv1_stride_t:int=1,
                 pool1_stride_s:int=2, pool1_stride_t:int=1, with_pool1:bool=True, with_pool2:bool=True,
                 style:str='pytorch', frozen_stages:int=-1, inflate:Sequence[int]=(1,1,1,1), inflate_style:str='3x1x1',
                 conv_cfg:dict=dict(type='Conv3d'), norm_cfg:dict=dict(type='BN3d', requires_grad=True),
                 act_cfg:dict=dict(type='ReLU', inplace=True), norm_eval:bool=False, with_cp:bool=False, 
                 non_local:Sequence[int]=(0,0,0,0), non_local_cfg:dict=dict(), zero_init_residual:bool=True, 
                 init_cfg:Optional[Union[dict, list[dict]]]=None,**kwargs)->None:
        super().__init__()
        assert depth in self.arch_settings, f"Invalid depth {depth} for resnet"
        self.depth=depth
        self.pretrained=pretrained
        self.pretrained2d=pretrained2d
        self.in_channels=in_channels
        self.base_channels=base_channels
        self.num_stages=num_stages
        assert 1<=num_stages<=4, f"num_stages must be in range [1,4], but got {num_stages}"
        self.stage_blocks=stage_blocks
        self.out_indices=out_indices
        assert max(out_indices)<num_stages, f"{max(out_indices)=} must be less than {num_stages=}"
        self.spatial_strides=spatial_strides
        self.temporal_strides=temporal_strides
        self.dilations=dilations
        assert len(spatial_strides)==len(temporal_strides)==len(dilations)==num_stages
        if self.stage_blocks is not None: assert len(self.stage_blocks)==num_stages

        self.conv1_kernel=conv1_kernel
        self.conv1_stride_s=conv1_stride_s
        self.conv1_stride_t=conv1_stride_t
        self.pool1_stride_s=pool1_stride_s
        self.pool1_stride_t=pool1_stride_t
        self.with_pool1=with_pool1
        self.with_pool2=with_pool2
        self.style=style
        self.frozen_stages=frozen_stages
        self.stage_inflations=_ntuple(num_stages)(inflate) # repeat `inflate` for `num_stages` times
        self.non_local_stages=_ntuple(num_stages)(non_local)
        self.inflate_style=inflate_style
        self.conv_cfg=conv_cfg
        self.norm_cfg=norm_cfg
        self.act_cfg=act_cfg
        self.norm_eval=norm_eval
        self.with_cp=with_cp
        self.zero_init_residual=zero_init_residual
        
        self.block, stage_blocks=self.arch_settings
        if self.stage_blocks is None: self.stage_blocks=stage_blocks[:num_stages]
        self.inplanes=self.base_channels
        self.non_local_cfg=non_local_cfg

        self._make_stem_layer()

        self.res_layers=[]
        lateral_inplanes=getattr(self, 'lateral_inplanes', [0,0,0,0])
        for i, num_blocks in enumerate(self.stage_blocks):
            spatial_stride=spatial_strides[i]
            temporal_stride=temporal_strides[i]
            dilation=dilations[i]
            planes=self.base_channels * 2**i
            res_layer=self.make_res_layer(self.block, self.inplanes+lateral_inplanes[i], planes, num_blocks, 
                                          spatial_stride=spatial_stride, temporal_stride=temporal_stride,
                                          dilation=dilation, style=self.style, norm_cfg=self.norm_cfg,
                                          conv_cfg=self.conv_cfg, act_cfg=self.act_cfg, non_local=self.non_local_stages[i],
                                          non_local_cfg=self.non_local_cfg, inflate=self.stage_inflations[i],
                                          inflate_style=self.inflate_style, with_cp=with_cp, **kwargs)
            self.inplanes=planes*self.block.expansion
            layer_name=f'layer{i+1}'
            self.add_module(layer_name, res_layer)
            self.res_layers.append(layer_name)

        self.feat_dim=self.block.expansion * self.base_channels * 2**(len(self.stage_blocks)-1)

    def _make_stem_layer(self)->None:
        """Construct the stem layers consists of a conv+norm+act module and a pooling layer"""
        self.conv1=ConvModule(self.in_channels, self.base_channels, kernel_size=self.conv1_kernel,
                             stride=(self.conv1_stride_t, self.conv1_stride_s, self.conv1_stride_s),
                             padding=tuple([(k-1)//2 for k in _triple(self.conv1_kernel)]),
                             bias=False, conv_cfg=self.conv_cfg, norm_cfg=self.norm_cfg, act_cfg=self.act_cfg)
        self.max_pool=nn.MaxPool3d(kernel_size=(1,3,3), stride=(self.pool1_stride_t, self.pool1_stride_s,
                                                               self.pool1_stride_s), padding=(0,1,1))
        self.pool2=nn.MaxPool3d(kernel_size=(2,1,1), stride=(2,1,1))

    @staticmethod
    def make_res_layer(block:nn.Module, inplanes:int, planes:int, blocks:int, spatial_stride:Union[int, Sequence[int]]=1,
                      temporal_stride:Union[int, Sequence[int]]=1, dilation:int=1, style:str='pytorch',
                      inflate:Union[int, Sequence[int]]=1, inflate_style:str='3x1x1', non_local:Union[int, Sequence[int]]=0,
                      non_local_cfg:dict=dict(), norm_cfg:Optional[dict]=None, act_cfg:Optional[dict]=None,
                      conv_cfg:Optional[dict]=None, with_cp:bool=False, **kwargs)->nn.Module:
        """Build residual layer for ResNet3d
        Args:
            block (nn.Module): Residual module to be built.
            inplanes (int): Number of channels for the input feature in each block
            planes (int): Number of channels for the output feature in each block
            blocks (int): Number of residual blocks
            spatial_stride (int|Sequence[int]): Spatial strides in residual and conv layers. Default to 1
            temporal_stride (int|Sequence[int]): Temporal stride in residual and conv layers. Default to 1
            dilation (int): Spacing between kernel elements. Default to 1
            style (str): 'pytorch' or 'caffe'. If set to 'pytorch', the stride-twi layer is the 3x3 conv layer;
                otherwise, the stride-two layer is the first 1x1 conv layer. Default to 'pytorch'
            inflate (int|Sequence[int]): Whether to inflate each block. Default to 1
            inflate_style (str): '3x1x1' or '3x3x3' determining the kernel sizes and padding strides for conv1 and conv2
                in each block. Default to '3x1x1'
            non_local (int|Sequence[int]): Whether to apply non-local module in the corresponding block of each stages. 
                Default to 0
            non_local_cfg (dict): Config for non-local module. Default to dict()
            conv_cfg (dict, optional): Config for conv layers. Default to None
            norm_cfg (dict, optional): Config for norm layers. Default to None
            act_cfg (dict, optional): Config for activate layers. Default to None.
            with_cp (bool, optional): Whether to use checkpoint. Using checkpoint will save some memory while slowing down 
                the training speed. Default to False
        Returns:
            (nn.Module): A residual layer for the given config
        """
        inflate=inflate if not isinstance(inflate, int) else (inflate,)*blocks
        non_local=non_local if not isinstance(non_local, int) else (non_local,)*blocks
        assert len(inflate)==len(non_local)==blocks
        downsample=None
        if spatial_stride!=1 or inplanes!=planes*block.expansion:
            downsample=ConvModule(inplanes, planes*block.expansion, kernel_size=1, 
                                  stride=(temporal_stride, spatial_stride, spatial_stride),
                                 bias=False, conv_cfg=conv_cfg, norm_cfg=norm_cfg,act_cfg=None)
        layers=[]
        layers.append(
            block(inplanes, planes, spatial_stride=spatial_stride, temporal_stride=temporal_stride, dilation=dilation,
                 downsample=downsample, style=style, inflate=(inflate[0]==1), inflate_style=inflate_style, non_local=(non_local[0]==1),
                 non_local_cfg=non_local_cfg, norm_cfg=norm_cfg, conv_cfg=conv_cfg, act_cfg=act_cfg, with_cp=with_cp, **kwargs)
        )
        inplanes=planes*block.expansion
        for i in range(1, blocks):
            layers.append(
                block(inplanes, planes, spatial_stride=1, temporal_stride=1, dilation=dilation, style=style, inflate=(inflate[i]==1),
                     inflate_style=inflate_style, non_local=(non_local[i]==1), non_local_cfg=non_local_cfg, norm_cfg=norm_cfg,
                     conv_cfg=conv_cfg, act_cfg=act_cfg, with_cp=with_cp, **kwargs)
            )
        return nn.Sequential(*layer)

(5, 5, 5, 5)